# Allen Mouse Connectivity Atlas ETL

Register injection experiments, experiment metadata, injection-site coordinates, brain regions and summary-region projection density. One `DataItem` represents an injection-defined population, **not a cell**. Cre lines remain experiment metadata, not cell-type assignments.

Inputs are the completed Parquet downloads in `data/Allen Mesoscale`. No AllenSDK calls or downloads are made here. [AllenSDK documentation](https://allensdk.readthedocs.io/en/latest/connectivity.html); [unionize definitions](https://allensdk.readthedocs.io/en/latest/unionizes.html); [Oh et al., 2014](https://doi.org/10.1038/nature13186).

**Release destination:** `scratch/ccm_v1.0.1_data_202611_v3`. After source validation, copy `ccm_v1.0.1_data_202611_v2` if v3 does not exist, then write only the mesoscale additions. This notebook deliberately overrides the configured output root so a configuration still pointing to v2 cannot modify it. Reruns target v3; the baseline and source downloads remain unchanged.

The annotation/template NRRDs remain supporting source assets; this ETL does not need to load or convert them. Full unionize metrics, voxel-level projection images, embeddings and region-to-region aggregation are deferred.

In [ ]:
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake
from polars.testing import assert_frame_equal

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import write_models, write_projection_matrix
from connects_common_connectivity.io.arrow_utils import (
    build_cell_feature_matrix_schema,
    models_to_table,
)
from connects_common_connectivity.io.path_spec import MODEL_TABLE_PATHS, WIDE_PAYLOAD_PATHS
from connects_common_connectivity.models import (
    AnatomicalOrientation,
    BrainRegion,
    BrainRegionAssociation,
    CellFeatureDefinition,
    CellFeatureMatrix,
    CellFeatureSet,
    DataItem,
    DataItemDataSetAssociation,
    DataSet,
    Default2DView,
    Laterality,
    LocationType,
    Modality,
    ProjectionMeasurementMatrix,
    ProjectionMeasurementType,
    ReferenceSpace,
    ReferenceSpaceReference,
    SignedAxis,
    SpatialLocation,
    Unit,
)


In [ ]:
REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'pyproject.toml').is_file()
)
INPUT_ROOT = REPO_ROOT / 'data/Allen Mesoscale'
BASE_ROOT = REPO_ROOT / 'scratch/ccm_v1.0.1_data_202611_v2'
OUTPUT_ROOT = REPO_ROOT / 'scratch/ccm_v1.0.1_data_202611_v3'
SETTINGS = get_settings().model_copy(update={'output_root': OUTPUT_ROOT})
PROJECT_ID = 'allen_mesoscale'
DATASET_ID = 'allen_mesoscale_connectivity'
FEATURE_SET_ID = 'allen_mesoscale_experiment_metadata'
REFERENCE_SPACE_ID = 'ccfv3_2017_microns'
PUBLICATION = 'https://doi.org/10.1038/nature13186'
SUMMARY_SET_ID = 167587189
CCF_EXTENT_UM = np.array([13200.0, 8000.0, 11400.0])
MIDLINE_UM = CCF_EXTENT_UM[2] / 2
MIDLINE_ISSUE = 'https://github.com/AllenInstitute/ConnectsCommonConnectivity/issues/91'
assert INPUT_ROOT.is_dir(), 'Complete the mesoscale download first.'
assert (BASE_ROOT / 'dataset' / '_delta_log').is_dir(), 'The v2 baseline is required.'
assert OUTPUT_ROOT.resolve() != BASE_ROOT.resolve(), 'Never write into the v2 baseline.'
assert not OUTPUT_ROOT.is_symlink(), 'The v3 destination must be an independent directory.'
assert not SETTINGS.dry_run, 'This notebook includes direct Delta writes; dry_run is unsupported.'
print('Input:', INPUT_ROOT, '\nBaseline:', BASE_ROOT, '\nOutput:', OUTPUT_ROOT)


## Load and validate the completed sources

Retain every experiment, including those without a reported transgenic line. Use source structure IDs as region identities. Summary structures (set 167587189) define the matrix column order; acronyms are only used to decode the downloaded matrix columns.

In [ ]:
experiments = pd.read_parquet(INPUT_ROOT / 'experiments.parquet').sort_values('id')
structures = pd.read_parquet(INPUT_ROOT / 'structures.parquet').sort_values('graph_order')
summary = pd.read_parquet(INPUT_ROOT / 'summary_structures.parquet').sort_values('graph_order')
downloaded = pd.read_parquet(INPUT_ROOT / 'projection_matrix_summary_structures.parquet')
for name, frame, key in [
    ('experiments', experiments, 'id'), ('structures', structures, 'id'),
    ('summary', summary, 'id'), ('downloaded matrix', downloaded, 'experiment_id'),
]:
    assert len(frame) and frame[key].notna().all() and frame[key].is_unique, name
assert summary['acronym'].is_unique, 'Summary acronyms must identify source columns uniquely.'
assert set(downloaded['experiment_id']) == set(experiments['id']), 'Incomplete matrix cohort.'
assert set(summary['id']) <= set(structures['id']), 'Unknown summary regions.'
assert all(SUMMARY_SET_ID in ids for ids in summary['structure_set_ids'])
pd.testing.assert_frame_equal(
    summary.set_index('id')[['acronym', 'name']],
    structures.set_index('id').loc[summary['id'], ['acronym', 'name']],
)
experiment_ids = experiments['id'].tolist()
item_ids = [str(eid) for eid in experiment_ids]
region_ids = summary['id'].astype(str).tolist()
experiments = experiments.set_index('id', drop=False)
downloaded = downloaded.set_index('experiment_id').loc[experiment_ids]
unionize_paths = [INPUT_ROOT / 'structure_unionizes' / f'experiment_{eid}.parquet'
                  for eid in experiment_ids]
missing = [path.name for path in unionize_paths if not path.is_file()]
assert not missing, f'Missing unionize files: {missing[:10]}'
coordinates = experiments[['injection_x', 'injection_y', 'injection_z']].to_numpy(dtype=float)
assert np.isfinite(coordinates).all(), 'Injection coordinates must be finite.'
assert ((coordinates >= 0) & (coordinates < CCF_EXTENT_UM)).all(), 'Outside CCF bounds.'
print(f'{len(item_ids):,} experiments; {len(structures):,} regions; {len(region_ids)} targets')


In [ ]:
unionizes = (
    pl.scan_parquet(unionize_paths)
    .filter(
        ~pl.col('is_injection') & pl.col('hemisphere_id').is_in([1, 2])
        & pl.col('structure_id').is_in(summary['id'].tolist())
    )
    .select('experiment_id', 'structure_id', 'hemisphere_id', 'projection_density')
    .collect()
    .to_pandas()
)
keys = ['experiment_id', 'structure_id', 'hemisphere_id']
assert not unionizes.duplicated(keys).any(), 'Duplicate non-injection unionizes.'
assert set(unionizes['experiment_id']) == set(experiment_ids), 'Incomplete unionize cohort.'
expected_columns = [f'{acronym}_{side}' for side in ('right', 'left')
                    for acronym in summary['acronym']]
assert downloaded.columns.tolist() == expected_columns, 'Unexpected source matrix columns.'
absolute_projections = {}
for hemisphere_id, side in [(1, 'left'), (2, 'right')]:
    frame = unionizes.loc[unionizes['hemisphere_id'] == hemisphere_id].pivot(
        index='experiment_id', columns='structure_id', values='projection_density',
    ).reindex(index=experiment_ids, columns=summary['id'])
    source = downloaded[[f'{acronym}_{side}' for acronym in summary['acronym']]]
    np.testing.assert_allclose(frame.to_numpy(), source.to_numpy(), rtol=0, atol=0,
                               equal_nan=True, err_msg=f'{side} matrix disagrees with unionizes')
    values = frame.to_numpy()
    assert (np.isnan(values) | (np.isfinite(values) & (values >= 0) & (values <= 1))).all()
    frame.index = pd.Index(item_ids, name='id')
    frame.columns = region_ids
    absolute_projections[side] = frame
del unionizes
print('Both downloaded hemispheres match non-injection unionizes exactly, including missingness.')


## Injection-relative laterality, with an explicit midline exception

Source coordinates are CCF micrometers: x=anterior-to-posterior, y=dorsal-to-ventral, z=subject-left-to-right. The 25-um atlas has shape (528, 320, 456); its ML extent is 11,400 um and midline is z=5,700 um. Use the **injection-site coordinate**, not the dominant injection volume, to choose the ipsilateral hemisphere. Injections can span both hemispheres; this is a site-relative convention, not a claim that the whole injection is unilateral.

In the November 2026 download, 2,977 experiments have a definite side. **Fifteen have z exactly at the midline (13 retina, 1 IMD, 1 CS).** Their coordinates cannot define ipsi/contra. Retain these experiments and their asymmetric measurements in two separate `_midline_left` / `_midline_right` matrices with `laterality=UNKNOWN`. Their IDs and descriptions carry the absolute side until [#91](https://github.com/AllenInstitute/ConnectsCommonConnectivity/issues/91) adds first-class support. Do not pool their hemispheres, guess a side, or append hemisphere suffixes to region IDs.

The other experiments appear once in each ipsi/contra matrix. All matrices contain **projection density outside the injection site** (`is_injection=False`), not projection intensity or injection-volume-normalized projection volume. `SIGNAL_INTENSITY` is the current broad tracer-signal enum, used with `RATIO` and explicit descriptions because there is no density-specific measurement enum.

In [ ]:
site_z = pd.Series(coordinates[:, 2], index=item_ids)
injection_side = pd.Series('midline', index=item_ids, name='injection_hemisphere')
injection_side.loc[site_z < MIDLINE_UM] = 'left'
injection_side.loc[site_z > MIDLINE_UM] = 'right'
lateral_ids = injection_side.index[injection_side != 'midline'].tolist()
midline_ids = injection_side.index[injection_side == 'midline'].tolist()
right_site = injection_side.loc[lateral_ids].eq('right').to_numpy()[:, None]
left = absolute_projections['left'].loc[lateral_ids].to_numpy()
right = absolute_projections['right'].loc[lateral_ids].to_numpy()
projections = {}
projection_specs = []
for suffix, values, laterality in [
    ('ipsi', np.where(right_site, right, left), Laterality.IPSILATERAL),
    ('contra', np.where(right_site, left, right), Laterality.CONTRALATERAL),
]:
    if lateral_ids:
        matrix_id = f'{PROJECT_ID}_projection_density_summary_{suffix}'
        projections[matrix_id] = pd.DataFrame(values, index=lateral_ids, columns=region_ids)
        projection_specs.append((matrix_id, laterality,
                                 'Relative to the injection-site coordinate.'))
for side in ('left', 'right'):
    if midline_ids:
        matrix_id = f'{PROJECT_ID}_projection_density_summary_midline_{side}'
        projections[matrix_id] = absolute_projections[side].loc[midline_ids].copy()
        projection_specs.append((matrix_id, Laterality.UNKNOWN,
                                 f'Absolute {side} hemisphere for {len(midline_ids)} midline '
                                 f'injection sites; ipsi/contra is undefined. '
                                 f'See {MIDLINE_ISSUE}.'))
assert set(lateral_ids).isdisjoint(midline_ids)
assert set(lateral_ids) | set(midline_ids) == set(item_ids)
print(injection_side.value_counts().to_string())
print('Midline experiments:')
midline_details = experiments.loc[[int(eid) for eid in midline_ids],
                                  ['structure_abbrev', 'injection_x', 'injection_y', 'injection_z']]
print(midline_details.to_string())


## Brain regions and injection-region associations

Build the full source hierarchy, with string structure IDs and integer annotation labels. The summary subset is identified by the projection matrices' ordered region indices. Region associations mean **source-reported injection structures**, including secondary structures; the primary injection structure is retained separately in experiment features. They are not projection-target associations or region assignments recomputed from a single coordinate.

`BrainRegion` and `BrainRegionAssociation` have no registered generic writers yet. Validate their models, convert them with the shared Arrow helper, and write directly. Existing global region IDs must have identical content; never overwrite another atlas's colliding IDs. The new global table uses `brainregion/`, which is not yet in the shared path registry.

In [ ]:
structure_ids = set(structures['id'].astype(str))
paths = {str(row.id): [str(int(value)) for value in row.structure_id_path]
         for row in structures.itertuples(index=False)}
children = {region: [] for region in paths}
descendants = {region: [] for region in paths}
for region, path in paths.items():
    assert path and path[-1] == region and len(path) == len(set(path)), region
    assert set(path) <= structure_ids, f'Unknown ancestor for {region}'
    if len(path) > 1:
        children[path[-2]].append(region)
    for position, ancestor in enumerate(path[:-1]):
        assert paths[ancestor] == path[:position + 1], f'Inconsistent hierarchy: {region}'
        descendants[ancestor].append(region)
brain_regions = [
    BrainRegion(
        id=str(row.id), name=row.name, acronym=row.acronym,
        parent_identifier=paths[str(row.id)][-2] if len(paths[str(row.id)]) > 1 else None,
        child_identifiers=children[str(row.id)], descendants=descendants[str(row.id)],
        annotation_value=int(row.id),
        descendant_annotation_values=[int(value) for value in descendants[str(row.id)]],
        hex_color=''.join(f'{int(value):02X}' for value in row.rgb_triplet),
        term_set_name='Allen Mouse Brain Atlas structure graph 1',
    )
    for row in structures.itertuples(index=False)
]
associations = []
for row in experiments.itertuples(index=False):
    regions = [str(int(value)) for value in row.injection_structures]
    assert len(regions) == len(set(regions)) and set(regions) <= structure_ids
    assert str(row.primary_injection_structure) in regions, f'Missing primary region: {row.id}'
    associations.extend(BrainRegionAssociation(
        project_id=PROJECT_ID, dataitem_id=str(row.id), brainregion_id=region,
    ) for region in regions)
region_table = models_to_table(brain_regions)
association_table = models_to_table(associations)
print(f'{len(brain_regions):,} brain regions; {len(associations):,} injection-region associations')


## Experiment features

Despite its historical name, `CellFeatureMatrix.cell_index_column` references DataItems and can hold experiment metadata. Preserve null strain/Cre information as null, not a guessed category. Source IDs are strings; injection-structure lists are JSON arrays of source integer IDs. Coordinates are retained unchanged in micrometers, and injection volume is in mm^3. The derived `injection_hemisphere` explicitly records the coordinate-based left/right/midline convention.

In [ ]:
feature_info = {
    'gender': ('Source-reported specimen sex (AllenSDK gender field).', None),
    'specimen_name': ('Source specimen name.', None),
    'strain': ('Source strain; missing values remain null.', None),
    'transgenic_line': ('Reported Cre/transgenic line; null is not a cell-type assignment.', None),
    'transgenic_line_id': ('Allen source identifier of the reported transgenic line.', None),
    'product_id': ('Allen source product identifier.', None),
    'primary_injection_structure': ('Primary injection region (a BrainRegion ID).', None),
    'injection_structures': ('JSON array of all source-reported injection structure IDs.', None),
    'injection_volume': ('Source injection volume, without ETL normalization.', 'mm^3'),
    'injection_x': ('Source injection-site CCF anterior-to-posterior coordinate.', 'micrometers'),
    'injection_y': ('Source injection-site CCF dorsal-to-ventral coordinate.', 'micrometers'),
    'injection_z': ('Source injection-site CCF subject-left-to-right coordinate.', 'micrometers'),
    'injection_hemisphere': ('Coordinate-based side: left below z=5700 um, right above, '
                             'midline at equality; not the entire injection footprint.', None),
}
feature_wide = experiments.drop(columns='id').copy()
feature_wide.index = pd.Index(item_ids, name='id')
for column in ['transgenic_line_id', 'product_id', 'primary_injection_structure']:
    feature_wide[column] = feature_wide[column].astype('Int64').astype('string')
feature_wide['injection_structures'] = feature_wide['injection_structures'].map(
    lambda values: json.dumps([int(value) for value in values])
)
feature_wide['injection_hemisphere'] = injection_side
feature_wide = feature_wide[list(feature_info)].reset_index()
feature_wide.insert(1, 'project_id', PROJECT_ID)
feature_wide.insert(2, 'feature_set_id', FEATURE_SET_ID)
numeric_features = {'injection_volume', 'injection_x', 'injection_y', 'injection_z'}
feature_definitions = []
for column, (description, unit) in feature_info.items():
    if column in numeric_features:
        feature_wide[column] = feature_wide[column].astype(float)
        dtype = '<f8'
    else:
        feature_wide[column] = feature_wide[column].astype('string')
        width = max((len(value) for value in feature_wide[column].dropna()), default=1)
        dtype = f'<U{max(1, width)}'
    feature_definitions.append(CellFeatureDefinition(
        id=column, description=description, unit=unit, data_type=dtype,
        project_id=PROJECT_ID, feature_set_id=FEATURE_SET_ID,
    ))
feature_set = CellFeatureSet(
    id=FEATURE_SET_ID, project_id=PROJECT_ID,
    description='Allen mesoscale injection-experiment metadata; rows are populations, not cells.',
    feature_definition_ids=list(feature_info),
    extraction_method='AllenSDK experiments.parquet; side derived from injection_z at 5700 um.',
)
feature_schema = build_cell_feature_matrix_schema(feature_set, feature_definitions)
feature_table = pa.Table.from_pandas(feature_wide, schema=feature_schema, preserve_index=False)


## Prepare v3 and register experiments

Only now, after source validation and preparation, create the output copy if needed. `copytree` copies Delta transaction logs and payloads without hard-linking mutable files. All registered-model writes use the explicit v3 settings; direct writes use v3 paths and project-scoped predicates. Atlas writes only insert new, non-conflicting global IDs.

In [ ]:
if not OUTPUT_ROOT.exists():
    shutil.copytree(BASE_ROOT, OUTPUT_ROOT)
    print('Copied v2 to', OUTPUT_ROOT)
else:
    for path in BASE_ROOT.iterdir():
        if path.is_dir() and (path / '_delta_log').is_dir():
            assert (OUTPUT_ROOT / path.name / '_delta_log').is_dir(), f'Incomplete v3: {path.name}'
    print('Using existing v3:', OUTPUT_ROOT)
brain_region_path = OUTPUT_ROOT / 'brainregion'
if DeltaTable.is_deltatable(str(brain_region_path)):
    existing_regions = pl.read_delta(brain_region_path)
    incoming_regions = pl.from_arrow(region_table)
    overlap = existing_regions.filter(pl.col('id').is_in(list(structure_ids)))
    assert_frame_equal(
        overlap.select(incoming_regions.columns).sort('id'),
        incoming_regions.filter(pl.col('id').is_in(overlap['id'].to_list())).sort('id'),
        check_dtypes=False,
    )
    DeltaTable(str(brain_region_path)).merge(
        region_table, predicate='target.id = source.id',
        source_alias='source', target_alias='target',
    ).when_not_matched_insert_all().execute()
else:
    write_deltalake(brain_region_path, region_table, mode='error')

reference_space = ReferenceSpace(
    id=REFERENCE_SPACE_ID, project_id=PROJECT_ID, name='Allen CCFv3 2017 micrometer coordinates',
    description='AllenSDK injection coordinates, unchanged; x=AP, y=DV, z=subject left-to-right. '
                'Atlas extent 13200 x 8000 x 11400 um; ML midline z=5700 um.',
    unit=Unit.MICRONS_LENGTH,
    anatomical_orientation=AnatomicalOrientation(
        anterior_to_posterior=SignedAxis.PLUS_X,
        dorsal_to_ventral=SignedAxis.PLUS_Y,
        subject_left_to_right=SignedAxis.PLUS_Z,
    ),
    default_2d_view=Default2DView(left_to_right=SignedAxis.PLUS_Z,
                                bottom_to_top=SignedAxis.MINUS_Y),
)
dataset = DataSet(
    id=DATASET_ID, project_id=PROJECT_ID,
    name='Allen Mouse Connectivity Atlas - injection experiments',
    modality=Modality.TRACER, publication=PUBLICATION,
    default_reference_space=ReferenceSpaceReference(project_id=PROJECT_ID, id=REFERENCE_SPACE_ID),
)
items = [DataItem(id=str(row.id), project_id=PROJECT_ID,
                  name=f'Allen mesoscale experiment {row.id}: {row.structure_abbrev}')
         for row in experiments.itertuples(index=False)]
memberships = [DataItemDataSetAssociation(
    project_id=PROJECT_ID, dataitem_id=eid, dataset_id=DATASET_ID,
) for eid in item_ids]
locations = [SpatialLocation(
    project_id=PROJECT_ID, dataitem_id=eid, reference_space=REFERENCE_SPACE_ID,
    location_type=LocationType.INJECTION_SITE,
    description='Source-reported injection-site point, not the full injection footprint.',
    x=float(x), y=float(y), z=float(z),
) for eid, (x, y, z) in zip(item_ids, coordinates)]
for rows in [[reference_space], [dataset], items, memberships, locations]:
    print(write_models(rows, settings=SETTINGS))
association_path = OUTPUT_ROOT / MODEL_TABLE_PATHS['BrainRegionAssociation']
write_deltalake(association_path, association_table, mode='overwrite',
                predicate=f"project_id = '{PROJECT_ID}'", partition_by=['project_id'])


In [ ]:
print(write_models(feature_set, settings=SETTINGS))
print(write_models(feature_definitions, settings=SETTINGS))
feature_path = OUTPUT_ROOT / WIDE_PAYLOAD_PATHS['cell_features'] / FEATURE_SET_ID
write_deltalake(feature_path, feature_table, mode='overwrite',
                predicate=f"project_id = '{PROJECT_ID}'",
                partition_by=['project_id', 'feature_set_id'])
feature_matrix = CellFeatureMatrix(
    id=FEATURE_SET_ID, feature_set_id=FEATURE_SET_ID, project_id=PROJECT_ID,
    cell_index_column='id', parquet_path=feature_path.relative_to(OUTPUT_ROOT).as_posix(),
)
print(write_models(feature_matrix, settings=SETTINGS))


## Write projection matrices

Follow the BARseq notebook's Delta-backed wide payload convention. Metadata and payloads share the projection table directory, but each matrix has its own payload subdirectory and release-relative pointer. Numeric values and missingness are preserved. Coverage includes only regions with a positive measured value; missing values do not count as signal.

In [ ]:
projection_paths = {}
for matrix_id, laterality, side_note in projection_specs:
    frame = projections[matrix_id]
    wide = frame.rename_axis('id').reset_index()
    wide.insert(1, 'project_id', PROJECT_ID)
    wide.insert(2, 'dataset_id', DATASET_ID)
    schema = pa.schema([
        pa.field('id', pa.string(), nullable=False),
        pa.field('project_id', pa.string(), nullable=False),
        pa.field('dataset_id', pa.string(), nullable=False),
        *[pa.field(region, pa.float64(), nullable=True) for region in region_ids],
    ])
    payload_path = OUTPUT_ROOT / MODEL_TABLE_PATHS['ProjectionMeasurementMatrix'] / matrix_id
    matrix = ProjectionMeasurementMatrix(
        id=matrix_id, project_id=PROJECT_ID,
        description=('AllenSDK projection_density = sum_projection_pixels / sum_pixels; '
                     'non-injection pixels only (is_injection=False), anterograde OUTPUT. '
                     f'Brain - Summary Structures set {SUMMARY_SET_ID}. {side_note} '
                     'SIGNAL_INTENSITY is a schema compatibility category, not '
                     'AllenSDK projection_intensity. Missing measurements remain null.'),
        measurement_type=ProjectionMeasurementType.SIGNAL_INTENSITY,
        modality=Modality.TRACER, unit=Unit.RATIO, laterality=laterality,
        region_index=region_ids, data_item_index=frame.index.tolist(),
        values=payload_path.relative_to(OUTPUT_ROOT).as_posix(),
    )
    write_deltalake(payload_path, pa.Table.from_pandas(wide, schema=schema, preserve_index=False),
                    mode='overwrite', predicate=f"project_id = '{PROJECT_ID}'",
                    partition_by=['project_id'])
    print(write_projection_matrix(matrix, frame.gt(0).to_numpy(), settings=SETTINGS))
    projection_paths[matrix_id] = payload_path


## Read-back checks

Verify identities, memberships, coordinates, region associations, feature values and all projection values after writing. These checks are safe to repeat and assert the exact cohort and matrix axes, not just that files exist.

In [ ]:
project_scope = pl.col('project_id') == PROJECT_ID
expected_rows = {
    'DataSet': 1, 'DataItem': len(item_ids), 'DataItemDataSetAssociation': len(item_ids),
    'ReferenceSpace': 1, 'SpatialLocation': len(item_ids),
    'BrainRegionAssociation': len(associations), 'CellFeatureSet': 1,
    'CellFeatureDefinition': len(feature_definitions), 'CellFeatureMatrix': 1,
    'ProjectionMeasurementMatrix': len(projection_specs),
}
for model, count in expected_rows.items():
    stored = pl.read_delta(OUTPUT_ROOT / MODEL_TABLE_PATHS[model]).filter(project_scope)
    assert stored.height == count, (model, stored.height, count)
    print(model, stored.height)
for model, expected, sort_keys in [
    ('DataSet', [dataset], ['id']), ('DataItem', items, ['id']),
    ('DataItemDataSetAssociation', memberships, ['dataitem_id']),
    ('ReferenceSpace', [reference_space], ['id']),
    ('SpatialLocation', locations, ['dataitem_id']),
    ('BrainRegionAssociation', associations, ['dataitem_id', 'brainregion_id']),
    ('CellFeatureSet', [feature_set], ['id']),
    ('CellFeatureDefinition', feature_definitions, ['id']),
    ('CellFeatureMatrix', [feature_matrix], ['id']),
]:
    expected_frame = pl.from_arrow(models_to_table(expected))
    stored = pl.read_delta(OUTPUT_ROOT / MODEL_TABLE_PATHS[model]).filter(project_scope)
    assert_frame_equal(stored.select(expected_frame.columns).sort(sort_keys),
                       expected_frame.sort(sort_keys), check_dtypes=False)
assert_frame_equal(
    pl.read_delta(brain_region_path).filter(pl.col('id').is_in(list(structure_ids))).sort('id'),
    pl.from_arrow(region_table).sort('id'), check_dtypes=False,
)
assert_frame_equal(pl.read_delta(feature_path).filter(project_scope)
                   .select(feature_table.column_names).sort('id'),
                   pl.from_arrow(feature_table).sort('id'), check_dtypes=False)
stored_matrices = pl.read_delta(
    OUTPUT_ROOT / MODEL_TABLE_PATHS['ProjectionMeasurementMatrix'],
).filter(project_scope)
for matrix_id, laterality, _ in projection_specs:
    frame = projections[matrix_id]
    stored = pl.read_delta(projection_paths[matrix_id]).filter(project_scope)
    assert stored.height == len(frame) and stored['id'].n_unique() == len(frame)
    assert stored['dataset_id'].unique().to_list() == [DATASET_ID]
    ordered = stored.to_pandas().set_index('id').loc[frame.index, region_ids]
    np.testing.assert_allclose(ordered.to_numpy(), frame.to_numpy(), rtol=0, atol=0, equal_nan=True)
    metadata = stored_matrices.filter(pl.col('id') == matrix_id).row(0, named=True)
    assert metadata['data_item_index'] == frame.index.tolist()
    assert metadata['region_index'] == region_ids
    assert metadata['region_coverage'] == frame.columns[frame.gt(0).any()].tolist()
    assert metadata['laterality'] == laterality.value
    assert metadata['modality'] == Modality.TRACER.value
    assert metadata['unit'] == Unit.RATIO.value
    assert metadata['measurement_type'] == ProjectionMeasurementType.SIGNAL_INTENSITY.value
    assert OUTPUT_ROOT / metadata['values'] == projection_paths[matrix_id]
    print(matrix_id, frame.shape, 'verified')
print('Allen mesoscale ETL verified in', OUTPUT_ROOT)
